# Decoding Word Meaning: A WordNet Exploration of Synsets, Hypernyms, Similarity and Context

CSET346 – Natural Language Processing | Lab 2

MANAS SHARMA
S24CSEU2420

## Setup

In [1]:
import nltk
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('brown')

from nltk.corpus import wordnet as wn
from nltk.corpus import brown
import pandas as pd

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package brown to /root/nltk_data...
[nltk_data]   Package brown is already up-to-date!


## Q1: Synsets and Meanings of Selected Words

In [2]:
words_q1 = ["car", "bank", "bat", "dog", "light", "plant", "book", "teacher", "student", "computer"]

for w in words_q1:
    print(f"Word: {w}")
    for syn in wn.synsets(w):
        example = syn.examples()[0] if syn.examples() else "N/A"
        print(f"  {syn.name()} | {syn.definition()} | Example: {example}")
    print()

Word: car


  car.n.01 | a motor vehicle with four wheels; usually propelled by an internal combustion engine | Example: he needs a car to get to work
  car.n.02 | a wheeled vehicle adapted to the rails of railroad | Example: three cars had jumped the rails
  car.n.03 | the compartment that is suspended from an airship and that carries personnel and the cargo and the power plant | Example: N/A
  car.n.04 | where passengers ride up and down | Example: the car was on the top floor
  cable_car.n.01 | a conveyance for passengers or freight on a cable railway | Example: they took a cable car to the top of the mountain

Word: bank
  bank.n.01 | sloping land (especially the slope beside a body of water) | Example: they pulled the canoe up on the bank
  depository_financial_institution.n.01 | a financial institution that accepts deposits and channels the money into lending activities | Example: he cashed a check at the bank
  bank.n.03 | a long ridge or pile | Example: a huge bank of earth
  bank.n.04 | a

### Words with Multiple Synsets – Observations

In [3]:
multi_syn_words = sorted(words_q1, key=lambda w: len(wn.synsets(w)), reverse=True)[:3]

for w in multi_syn_words:
    print(f"{w} has {len(wn.synsets(w))} synsets")
    for syn in wn.synsets(w)[:3]:
        print(f"  {syn.name()}: {syn.definition()}")
    print()

light has 47 synsets
  light.n.01: (physics) electromagnetic radiation that can produce a visual sensation
  light.n.02: any device serving as a source of illumination
  light.n.03: a particular perspective or aspect of a situation

bank has 18 synsets
  bank.n.01: sloping land (especially the slope beside a body of water)
  depository_financial_institution.n.01: a financial institution that accepts deposits and channels the money into lending activities
  bank.n.03: a long ridge or pile

book has 15 synsets
  book.n.01: a written work or composition that has been published (printed on pages bound together)
  book.n.02: physical objects consisting of a number of pages bound together
  record.n.05: a compilation of the known facts regarding something or someone



## Q2: Hypernyms and Hierarchical Semantic Relationships

In [4]:
nouns_q2 = ["car", "dog", "computer", "bank", "book", "teacher", "student", "plant", "bird", "chair"]

rows = []
for w in nouns_q2:
    syn = wn.synsets(w, pos=wn.NOUN)[0]
    hypernyms = syn.hypernyms()
    hyp_names = [h.name() for h in hypernyms]
    rows.append({"Word": w, "Synset": syn.name(), "Direct Hypernyms": hyp_names})

df_hyper = pd.DataFrame(rows)
df_hyper

,Word,Synset,Direct Hypernyms
0,car,car.n.01,[motor_vehicle.n.01]
1,dog,dog.n.01,"[domestic_animal.n.01, canine.n.02]"
2,computer,computer.n.01,[machine.n.01]
3,bank,bank.n.01,[slope.n.01]
4,book,book.n.01,[publication.n.01]
5,teacher,teacher.n.01,[educator.n.01]
6,student,student.n.01,[enrollee.n.01]
7,plant,plant.n.01,[building_complex.n.01]
8,bird,bird.n.01,[vertebrate.n.01]
9,chair,chair.n.01,[seat.n.03]


In [5]:
def hypernym_chain(synset, levels=3):
    chain = []
    current = synset
    for _ in range(levels):
        hyps = current.hypernyms()
        if not hyps:
            break
        chain.append(hyps[0].name())
        current = hyps[0]
    return chain

for w in nouns_q2[:5]:
    syn = wn.synsets(w, pos=wn.NOUN)[0]
    print(f"{w} ({syn.name()}): {' -> '.join(hypernym_chain(syn))}")

car (car.n.01): motor_vehicle.n.01 -> self-propelled_vehicle.n.01 -> wheeled_vehicle.n.01
dog (dog.n.01): domestic_animal.n.01 -> animal.n.01 -> organism.n.01
computer (computer.n.01): machine.n.01 -> device.n.01 -> instrumentality.n.03
bank (bank.n.01): slope.n.01 -> geological_formation.n.01 -> object.n.01
book (book.n.01): publication.n.01 -> work.n.02 -> product.n.02


## Q3: WordNet Semantic Similarity Between Word Pairs

In [6]:
pairs = [
    ("car", "automobile"),
    ("dog", "cat"),
    ("dog", "animal"),
    ("dog", "computer"),
    ("bank", "money"),
    ("teacher", "student"),
    ("book", "novel"),
    ("light", "dark")
]

results = []
for w1, w2 in pairs:
    syn1 = wn.synsets(w1, pos=wn.NOUN)
    syn2 = wn.synsets(w2, pos=wn.NOUN)
    if syn1 and syn2:
        score = syn1[0].path_similarity(syn2[0])
    else:
        score = None
    results.append({"Word1": w1, "Word2": w2, "Synset1": syn1[0].name() if syn1 else None,
                     "Synset2": syn2[0].name() if syn2 else None, "Similarity": score})

df_sim = pd.DataFrame(results).sort_values(by="Similarity", ascending=False).reset_index(drop=True)
df_sim

,Word1,Word2,Synset1,Synset2,Similarity
0,car,automobile,car.n.01,car.n.01,1.000000
1,dog,animal,dog.n.01,animal.n.01,0.333333
2,dog,cat,dog.n.01,cat.n.01,0.200000
3,teacher,student,teacher.n.01,student.n.01,0.142857
4,dog,computer,dog.n.01,computer.n.01,0.090909
5,bank,money,bank.n.01,money.n.01,0.083333
6,light,dark,light.n.01,dark.n.01,0.062500
7,book,novel,book.n.01,novel.n.01,0.058824


### Observation
Similarity depends on the distance between synsets in the WordNet hypernym tree — closely related concepts sharing a near ancestor score high, while unrelated concepts (e.g., dog–computer) score low or near zero.

## Q4: Homonyms and Contextual Words

In [7]:
homonym_sentences = {
    "bank": [
        "I deposited money in the bank yesterday.",
        "We sat on the bank of the river and watched the sunset."
    ],
    "bat": [
        "The bat flew out of the cave at dusk.",
        "He hit the ball hard with the bat."
    ],
    "bark": [
        "The dog began to bark loudly at the stranger.",
        "The bark of the old oak tree was rough and grey."
    ],
    "light": [
        "Please turn off the light before you sleep.",
        "This bag is very light and easy to carry."
    ],
    "match": [
        "They watched an exciting football match last night.",
        "He lit the candle with a single match."
    ]
}

for word, sents in homonym_sentences.items():
    print(f"Word: {word}")
    for syn in wn.synsets(word)[:4]:
        print(f"  {syn.name()}: {syn.definition()}")
    print()

Word: bank
  bank.n.01: sloping land (especially the slope beside a body of water)
  depository_financial_institution.n.01: a financial institution that accepts deposits and channels the money into lending activities
  bank.n.03: a long ridge or pile
  bank.n.04: an arrangement of similar objects in a row or in tiers

Word: bat
  bat.n.01: nocturnal mouselike mammal with forelimbs modified to form membranous wings and anatomical adaptations for echolocation by which they navigate
  bat.n.02: (baseball) a turn trying to get a hit
  squash_racket.n.01: a small racket with a long handle used for playing squash
  cricket_bat.n.01: the club used in playing cricket

Word: bark
  bark.n.01: tough protective covering of the woody stems and roots of trees and other woody plants
  bark.n.02: a noise resembling the bark of a dog
  bark.n.03: a sailing ship with 3 (or more) masts
  bark.n.04: the sound made by a dog

Word: light
  light.n.01: (physics) electromagnetic radiation that can produce a 

In [8]:
contextual_clues = {
    "bank": ["deposited, money", "river, sat"],
    "bat": ["flew, cave", "hit, ball"],
    "bark": ["dog, loudly", "tree, rough"],
    "light": ["turn off, sleep", "bag, carry"],
    "match": ["football, watched", "candle, lit"]
}

intended_meanings = {
    "bank": ["financial institution", "land alongside a river"],
    "bat": ["flying nocturnal mammal", "sports equipment"],
    "bark": ["sound made by a dog", "outer covering of a tree"],
    "light": ["illumination device", "not heavy"],
    "match": ["sporting contest", "stick for lighting fire"]
}

final_rows = []
for word, sents in homonym_sentences.items():
    for i, sent in enumerate(sents):
        final_rows.append({
            "Word": word,
            "Sentence": sent,
            "Intended Meaning": intended_meanings[word][i],
            "Contextual Clue": contextual_clues[word][i]
        })

df_final = pd.DataFrame(final_rows)
df_final

,Word,Sentence,Intended Meaning,Contextual Clue
0,bank,I deposited money in the bank yesterday.,financial institution,"deposited, money"
1,bank,We sat on the bank of the river and watched th...,land alongside a river,"river, sat"
2,bat,The bat flew out of the cave at dusk.,flying nocturnal mammal,"flew, cave"
3,bat,He hit the ball hard with the bat.,sports equipment,"hit, ball"
4,bark,The dog began to bark loudly at the stranger.,sound made by a dog,"dog, loudly"
5,bark,The bark of the old oak tree was rough and grey.,outer covering of a tree,"tree, rough"
6,light,Please turn off the light before you sleep.,illumination device,"turn off, sleep"
7,light,This bag is very light and easy to carry.,not heavy,"bag, carry"
8,match,They watched an exciting football match last n...,sporting contest,"football, watched"
9,match,He lit the candle with a single match.,stick for lighting fire,"candle, lit"
